## LSTM을 이용한 시계열 예측 (Time Series Forecasting)

### 0. 시계열 예측

* 시계열 예측: 과거의 값들을 보고 **다음 값**을 예측하는 문제
    - Ex) 지난 24시간의 자전거 대여량 → 다음 1시간의 대여량
    - Ex) 지난 일주일의 기온 → 내일 기온
* 9-1에서 배운 **다대일(many-to-one)** 구조
    - 입력: 시퀀스 (과거 24개의 값), 출력: 값 하나 (다음 값)
* 이번 시간에는 같은 문제를 **선형 회귀**(4-1)와 **LSTM**(9-1)으로 각각 풀고 결과를 비교

### 1. 데이터 이해 및 전처리

#### 1) 데이터 로드

* 서울시 공공자전거(따릉이) **시간별 대여량** 데이터 (UCI Machine Learning Repository, Seoul Bike Sharing Demand)
    - 2017년 12월 1일 ~ 2018년 11월 30일, 365일 × 24시간 = 8,760개
    - 대여량 외에 기온, 습도, 강수량 등의 날씨 정보도 있지만 이번에는 **대여량만** 사용

In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
os.makedirs('./Data/bike', exist_ok=True)
urllib.request.urlretrieve('https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip', './Data/bike/seoul_bike.zip')
zipfile.ZipFile('./Data/bike/seoul_bike.zip').extractall('./Data/bike')   # 압축 해제 → SeoulBikeData.csv

In [ ]:
data = pd.read_csv('./Data/bike/SeoulBikeData.csv', encoding='cp1252')   # 열 이름에 °C 같은 특수문자가 있어 인코딩 지정
data.head()

In [ ]:
data = data[['Date', 'Hour', 'Rented Bike Count']]   # 필요한 열만 선택
data.columns = ['date', 'hour', 'count']              # 열 이름 변경
data['datetime'] = pd.to_datetime(data['date'], format='%d/%m/%Y') + pd.to_timedelta(data['hour'], unit='h')   # 날짜 + 시간
data.head()

In [ ]:
print(len(data))
data['count'].describe()

#### 2) 데이터 살펴보기

In [ ]:
# 1년 전체 대여량
plt.figure(figsize=(14, 4))
plt.plot(data['datetime'], data['count'], lw=0.5)
plt.xlabel('Date')
plt.ylabel('Rented bikes')
plt.show()

* 여름에 많고 겨울에 적음 (계절 패턴)
* 중간중간 0으로 떨어지는 날은 시스템 점검 등으로 운영하지 않은 날

In [ ]:
# 일주일(7일 × 24시간)만 확대해서 보기
week = data[(data['datetime'] >= '2018-06-04') & (data['datetime'] < '2018-06-11')]   # 월요일 ~ 일요일

plt.figure(figsize=(14, 4))
plt.plot(week['datetime'], week['count'], marker='o', ms=3)
plt.xlabel('Date')
plt.ylabel('Rented bikes')
plt.show()

In [ ]:
# 시간대별 평균 대여량
hourly_mean = data.groupby('hour')['count'].mean()

plt.figure(figsize=(8, 3))
plt.bar(hourly_mean.index, hourly_mean.values)
plt.xlabel('Hour')
plt.ylabel('Average rented bikes')
plt.xticks(range(0, 24))
plt.show()

* 하루 안에서도 **출근 시간(8시)과 퇴근 시간(18시)에 봉우리**가 두 개 있고, 새벽에는 거의 없음
* 평일과 주말의 모양이 다름
* 이런 패턴은 직선으로 표현하기 어려운 **비선형** 패턴 → 뒤에서 선형 회귀와 LSTM의 차이로 나타남

#### 3) 학습 데이터와 테스트 데이터 분리

* 시계열 데이터는 **시간 순서대로** 나눔: 앞부분(과거)으로 학습, 뒷부분(미래)으로 테스트
    - 9-1에서 본 것처럼 순차 데이터는 샘플이 서로 독립이 아님
    - 무작위로 섞어서 나누면 "미래를 보고 과거를 맞히는" 셈이 되어 성능이 실제보다 좋게 나옴
* 앞 300일(7,200시간)은 학습, 뒤 65일(1,560시간)은 테스트

In [ ]:
series = data['count'].values   # 대여량만 1차원 배열로

n_train = 24 * 300   # 300일
train = series[:n_train]
test = series[n_train:]
print(len(train), len(test))
print('학습 시작:', data['datetime'].iloc[0])
print('테스트 시작:', data['datetime'].iloc[n_train])

#### 4) 스케일링

* 대여량은 0 ~ 3,500 범위 → 신경망이 학습하기 좋도록 **0 ~ 1 사이로** 변환 (최소-최대 스케일링)
* 최솟값과 최댓값은 **학습 데이터에서만** 구함
    - 테스트 데이터의 값을 쓰면 미래 정보를 미리 보는 것

In [ ]:
min_value = train.min()
max_value = train.max()
print(min_value, max_value)

scaled = (series - min_value) / (max_value - min_value)   # 전체 시계열에 같은 기준 적용
print(scaled[:5])

#### 5) 슬라이딩 윈도우 (Sliding Window)

* 시계열 하나를 "입력 시퀀스 → 정답" 쌍 여러 개로 잘라야 지도 학습이 가능
* 윈도우(window): 입력으로 사용할 과거 값의 개수. 여기서는 24 (하루)

```
시계열:  x0  x1  x2  x3  ...  x23  x24  x25  x26 ...

샘플 1:  [x0  x1  x2  ...  x23]  →  x24
샘플 2:      [x1  x2  ...  x23  x24]  →  x25
샘플 3:          [x2  ...  x23  x24  x25]  →  x26
...
```
* 윈도우를 한 칸씩 밀면서 (슬라이딩) 샘플을 만듦

In [ ]:
def make_windows(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i + window])   # 과거 window개
        y.append(data[i + window])     # 바로 다음 값
    return np.array(X), np.array(y)

window = 24
X_all, y_all = make_windows(scaled, window)
print(X_all.shape, y_all.shape)

In [ ]:
# 첫 번째 샘플 확인: 입력 24개와 정답
print(X_all[0])
print(y_all[0])
print(scaled[24])   # 정답 = 시계열의 25번째 값

* 정답 `y_all[i]`는 시계열의 `i + window`번째 값
* 정답이 학습 구간(`n_train` 이전)에 있는 샘플은 학습용, 테스트 구간에 있는 샘플은 테스트용
    - 테스트 첫 샘플의 입력은 학습 구간의 마지막 24시간이지만, 맞혀야 할 정답(`n_train`번째 값)은 학습에 한 번도 쓰이지 않았으므로 문제 없음
    - 실제 예측 상황과 같음: 어제까지의 값은 알고 있는(학습에 쓴) 정보이고, 모르는 것은 오늘의 값뿐

In [ ]:
X_train, y_train = X_all[:n_train - window], y_all[:n_train - window]
X_test, y_test = X_all[n_train - window:], y_all[n_train - window:]
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

#### 6) 텐서 변환

* RNN의 입력 형태: `(batch_size, time_steps, input_size)` (9-1 참고)
    - 한 시점의 입력은 대여량 값 하나 → `input_size = 1`
    - `(샘플 수, 24)` → `(샘플 수, 24, 1)`로 차원 추가 (`unsqueeze`)
* 정답도 `(샘플 수,)` → `(샘플 수, 1)`로 맞춤

In [ ]:
X_train = torch.FloatTensor(X_train).unsqueeze(2)   # (7176, 24) → (7176, 24, 1)
y_train = torch.FloatTensor(y_train).unsqueeze(1)   # (7176,)   → (7176, 1)
X_test = torch.FloatTensor(X_test).unsqueeze(2)
y_test = torch.FloatTensor(y_test).unsqueeze(1)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

* 미니배치 학습을 위해 `DataLoader` 사용
    - 윈도우 **안의** 순서는 그대로이므로, 샘플(윈도우)**끼리**는 섞어도 됨

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

### 2. 선형 회귀로 예측

* 4-1에서 배운 선형 회귀: 입력의 **가중합**으로 예측
$$
\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_{24} x_{24} + b
$$
* 과거 24시간의 값 각각에 가중치를 곱해 더하는 모델 → `nn.Linear(24, 1)`
    - 입력이 `(배치 크기, 24, 1)`이므로 `nn.Flatten()`으로 `(배치 크기, 24)`로 펼친 뒤 전달

In [ ]:
linear_model = nn.Sequential(
    nn.Flatten(),        # (batch_size, 24, 1) → (batch_size, 24)
    nn.Linear(window, 1) # 24개의 가중치 + 편향 1개
)
print(linear_model)

**모델을 클래스로 만들기**

* `nn.Sequential`: 층을 순서대로 나열하면 입력이 그 순서대로 통과 → 간단하지만, 층 사이에서 다른 작업을 끼워 넣을 수 없음
    - Ex) LSTM은 출력이 두 개(`outputs`, `_status`)라서 그중 하나만 골라 다음 층에 넘겨야 함
* `nn.Module`을 상속한 **클래스**로 모델을 정의하면 자유롭게 작성 가능 (1-4 클래스 복습)
    - `__init__`: 사용할 층을 만들어 둠 (`Sequential`에 나열하던 것)
    - `forward`: 입력 `x`가 층을 어떤 순서로 지나는지 적음. `model(x)`를 호출하면 자동으로 실행됨
* 위의 선형 회귀 모델을 클래스로 다시 쓰면:

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, window):
        super(LinearModel, self).__init__()   # nn.Module의 초기화
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(window, 1)

    def forward(self, x):          # x: (batch_size, 24, 1)
        x = self.flatten(x)        # (batch_size, 24)
        return self.fc(x)          # (batch_size, 1)

In [ ]:
torch.manual_seed(0)

linear_model = LinearModel(window)   # nn.Sequential(nn.Flatten(), nn.Linear(window, 1)) 과 같은 모델
print(linear_model)

* 두 방식은 완전히 같은 모델이며, 학습·예측 코드도 똑같이 사용 가능
* 앞으로는 **클래스 방식**을 사용

**학습 함수**

* 선형 회귀와 LSTM을 **같은 코드**로 학습하기 위해 함수로 작성
* 손실 함수: MSE (회귀 문제), 옵티마이저: Adam

In [ ]:
def train(model, dataloader, epochs, lr=0.005):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}')

In [ ]:
train(linear_model, train_dataloader, epochs=40)

**예측 및 평가**

* 예측값은 0~1로 스케일링된 값 → 원래 단위(대)로 되돌린 뒤 평가
* 평가 지표: **MAE** (Mean Absolute Error, 평균 절대 오차) — 예측이 실제와 평균 몇 대 차이 나는지

In [ ]:
def inverse_scale(x):
    return x * (max_value - min_value) + min_value   # 0~1 → 원래 대여량

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def predict(model, X):
    model.eval()
    with torch.no_grad():
        y_pred = model(X)
    return inverse_scale(y_pred.squeeze(1).numpy())   # 원래 단위로

y_test_actual = inverse_scale(y_test.squeeze(1).numpy())   # 테스트 정답 (원래 단위)

In [ ]:
linear_pred = predict(linear_model, X_test)
mae_linear = mae(y_test_actual, linear_pred)
print(f'선형 회귀 MAE: {mae_linear:.1f}대')

In [ ]:
# 테스트 구간의 마지막 일주일(168시간) 비교
test_time = data['datetime'].iloc[n_train:]   # 테스트 구간의 시각

plt.figure(figsize=(14, 4))
plt.plot(test_time[-168:], y_test_actual[-168:], label='Actual', color='black')
plt.plot(test_time[-168:], linear_pred[-168:], label='Linear regression')
plt.xlabel('Date')
plt.ylabel('Rented bikes')
plt.legend()
plt.show()

* 대체로 따라가지만, 봉우리(출퇴근 시간)의 높이를 제대로 못 맞히고 **한 시간씩 늦게 따라가는** 모양
* 선형 회귀 가중치를 보면 대부분 **바로 직전 값**($x_{24}$)에 의존함을 알 수 있음

In [ ]:
weights = linear_model.fc.weight.detach().numpy().flatten()   # nn.Linear의 가중치 24개

plt.figure(figsize=(8, 3))
plt.bar(range(1, window + 1), weights)
plt.xlabel('Input position (1 = 24 hours ago, 24 = 1 hour ago)')
plt.ylabel('Weight')
plt.show()

### 3. LSTM으로 예측

* 9-1에서 배운 `nn.LSTM`을 사용한 다대일 모델. `LinearModel`과 같은 방식으로 클래스로 정의
    - `nn.LSTM(input_size=1, hidden_size, batch_first=True)` → 24개 시점을 차례로 읽으며 은닉 상태를 갱신
    - `forward`에서 마지막 시점의 출력 `outputs[:, -1, :]`만 꺼내 `nn.Linear`로 값 하나를 예측

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):                   # x: (batch_size, 24, 1)
        outputs, _status = self.lstm(x)     # outputs: (batch_size, 24, hidden_size)
        last = outputs[:, -1, :]            # 마지막 시점의 출력: (batch_size, hidden_size)
        return self.fc(last)                # (batch_size, 1)

In [ ]:
torch.manual_seed(0)

lstm_model = LSTMModel(input_size=1, hidden_size=64)
print(lstm_model)

In [ ]:
train(lstm_model, train_dataloader, epochs=40)

In [ ]:
lstm_pred = predict(lstm_model, X_test)
mae_lstm = mae(y_test_actual, lstm_pred)
print(f'선형 회귀 MAE: {mae_linear:.1f}대')
print(f'LSTM MAE: {mae_lstm:.1f}대')

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(test_time[-168:], y_test_actual[-168:], label='Actual', color='black')
plt.plot(test_time[-168:], linear_pred[-168:], label='Linear regression', alpha=0.7)
plt.plot(test_time[-168:], lstm_pred[-168:], label='LSTM')
plt.xlabel('Date')
plt.ylabel('Rented bikes')
plt.legend()
plt.show()

* LSTM은 출퇴근 시간의 봉우리를 훨씬 잘 따라감
* 같은 입력(과거 24시간)을 받았지만, 선형 회귀는 값들의 가중합만 가능한 반면 LSTM은 "지금이 하루 중 어느 시점인지"와 같은 **패턴을 은닉 상태에 기억**하면서 읽기 때문

### 4. 정리

* 시계열 예측 = 슬라이딩 윈도우로 "과거 시퀀스 → 다음 값" 샘플을 만든 뒤 **다대일** 모델로 학습
* 전처리에서 지켜야 할 것
    - 학습/테스트는 **시간 순서**로 분리
    - 스케일링 기준(최소·최대)은 **학습 데이터**에서만 계산
* 선형 회귀 vs LSTM
    - 선형 회귀: 과거 값들의 가중합 → 단순하고 빠르지만 비선형 패턴(하루 두 번의 봉우리)을 표현하기 어려움
    - LSTM: 시퀀스를 순서대로 읽으며 패턴을 기억 → 데이터가 충분하면 복잡한 패턴도 학습
    - 항상 LSTM이 이기는 것은 아님. 데이터가 적거나 패턴이 단순하면 선형 회귀가 더 나을 수도 있음

### 5. 연습문제

1. `nn.LSTM`을 `nn.RNN`, `nn.GRU`로 바꾸어 같은 조건으로 학습하고 MAE를 비교해 보세요. 9-1에서 배운 세 모델의 차이가 결과에 나타나나요?

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(RNNModel, self).__init__()
        # 여기에 코드를 작성하세요 (nn.RNN 사용)

    def forward(self, x):
        # 여기에 코드를 작성하세요
        pass


torch.manual_seed(0)
rnn_model = RNNModel(input_size=1, hidden_size=64)
train(rnn_model, train_dataloader, epochs=40)
rnn_pred = predict(rnn_model, X_test)
print(f'RNN MAE: {mae(y_test_actual, rnn_pred):.1f}대')

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(GRUModel, self).__init__()
        # 여기에 코드를 작성하세요 (nn.GRU 사용)

    def forward(self, x):
        # 여기에 코드를 작성하세요
        pass


torch.manual_seed(0)
gru_model = GRUModel(input_size=1, hidden_size=64)
train(gru_model, train_dataloader, epochs=40)
gru_pred = predict(gru_model, X_test)
print(f'GRU MAE: {mae(y_test_actual, gru_pred):.1f}대')